# 25 · Migration cutover and rollback

## The engineering question
How can a new representation or chunking pipeline catch up with changes made during backfill without reviving deleted data?

**Learning outcomes:** identify a source-log high-water mark; replay ordered updates and tombstones; recover an interrupted consumer; reconcile both indexes; cut over an alias; keep a rollback index current; enforce current permissions independently of either index.

This single-writer exercise uses an authoritative SQLite source journal with monotonic event sequence and compare-and-swap source revisions. Revisions are opaque strings, so alphabetical ordering cannot establish freshness. Each index has its own durable replay cursor and ingestion manifest. An interrupted remote update is replayed idempotently before advancing the cursor. Production designs require retention guarantees, worker leases, and a real transactional outbox/change feed.

Old and new indexes share the embedding model but change chunking from document-level to overlapping 12-word chunks. The same mechanism supports a model migration with a separately configured new encoder; aliases do not make incompatible query vectors safe. Queries use physical index objects after a drained service restart, so the query encoder and cache generation cannot lag behind an alias change.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("25", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from uuid import uuid4
from fastapi.testclient import TestClient
from qdrant_client import models as qm
from vector_masterclass.schema import Document
from vector_masterclass.store import VectorIndex
from vector_masterclass.ingestion import Ingestor
from vector_masterclass.journal import SourceJournal
from vector_masterclass.policy import PolicyStore, seed_demo_policy
from vector_masterclass.service import SearchService, create_app
from vector_masterclass.operations import switch_alias

prefix = "vmc_25_" + uuid4().hex[:8]
old, new = VectorIndex(prefix + "_old"), VectorIndex(prefix + "_new")
old.create()
new.create()
a = Ingestor(old, lesson.path / "old.sqlite", strategy="document", include_sparse=False)
b = Ingestor(
    new, lesson.path / "new.sqlite", strategy="fixed", words=12, overlap=3, include_sparse=False
)
journal = SourceJournal(lesson.path / "journal.sqlite")
policy = PolicyStore(lesson.path / "policy.sqlite")
docs = [
    Document(
        id="manual",
        revision="1",
        tenant="public",
        title="Manual",
        text="The Atlas router warranty lasts two years. Export the configuration from Settings then Backup then Export before a factory reset.",
    ),
    Document(
        id="retired",
        revision="1",
        tenant="public",
        title="Retired",
        text="Retired product instructions should be removed.",
    ),
    Document(
        id="private",
        revision="1",
        tenant="red",
        title="Private",
        text="The red tenant migration code is CEDAR.",
    ),
]
seed_demo_policy(policy, docs)
for doc in docs:
    journal.append(doc, expected_revision=None)
rows = []
alias = prefix + "_read"


def probe(index, phase):
    service = SearchService(index, policy)
    with TestClient(create_app(service)) as api:
        response = api.post(
            "/search",
            json={"query": "warranty migration code", "k": 20},
            headers={"Authorization": "Bearer demo-red-token"},
        )
        assert response.status_code == 200, response.text
        keys = {h["chunk"]["document_key"] for h in response.json()["results"]}
        rows.append(
            {
                "phase": phase,
                "collection": index.name,
                "sources": sorted(keys),
                "cache_hit": response.json()["cache_hit"],
                "policy_epoch": response.json()["policy_epoch"],
            }
        )
        return keys


try:
    journal.replay(a)
    journal.replay(b, stop_after=1)
    watermark, initial = journal.snapshot()
    updated = docs[0].model_copy(
        update={"revision": "2", "text": docs[0].text.replace("two years", "three years")}
    )
    deleted = docs[1].model_copy(update={"revision": "2", "active": False})
    journal.append(updated, expected_revision="1")
    journal.append(deleted, expected_revision="1")
    policy.put_source(updated)
    policy.put_source(deleted)
    policy.revoke_grant("alice", "red", "private")
    stale_rejected = False
    try:
        journal.append(docs[0], expected_revision="1")
    except ValueError:
        stale_rejected = True
    interrupted = False
    try:
        journal.replay(b, fail_before_checkpoint=True)
    except InterruptedError:
        interrupted = True
    # Close and reopen the new consumer, proving cursor/manifest durability.
    b.close()
    b = Ingestor(
        new, lesson.path / "new.sqlite", strategy="fixed", words=12, overlap=3, include_sparse=False
    )
    journal.replay(a)
    journal.replay(b)
    head, current = journal.snapshot()
    assert journal.cursor(a) == journal.cursor(b) == head
    assert a.reconcile()["consistent"] and b.reconcile()["consistent"]
    assert {(d.tenant, d.key) for d in a.current_documents()} == {
        (d.tenant, d.key) for d in b.current_documents()
    }
    switch_alias(old.client, alias, new.name)
    new_keys = probe(new, "cutover")
    # A later correction must reach BOTH versions before rollback is safe.
    corrected = updated.model_copy(
        update={"revision": "3", "text": updated.text + " Support hours are 09:00 to 17:00."}
    )
    journal.append(corrected, expected_revision="2")
    policy.put_source(corrected)
    journal.replay(b)
    journal.replay(a)
    switch_alias(old.client, alias, old.name)
    rollback_keys = probe(old, "rollback after catch-up")
    head, _ = journal.snapshot()
    assert journal.cursor(a) == journal.cursor(b) == head
    alias_target = next(
        x.collection_name for x in old.client.get_aliases().aliases if x.alias_name == alias
    )
    point_counts = [len(old.all_points()), len(new.all_points())]
    reconciled = a.reconcile()["consistent"] and b.reconcile()["consistent"]
finally:
    old.client.update_collection_aliases(
        [qm.DeleteAliasOperation(delete_alias=qm.DeleteAlias(alias_name=alias))]
    )
    a.close()
    b.close()
    journal.close()
    policy.close()
    old.drop()
    new.drop()
    old.close()
    new.close()
results = lesson.table(pd.DataFrame(rows))
display(results)
fig, ax = plt.subplots()
ax.bar(["old document chunks", "new 12-word chunks"], point_counts)
ax.set(ylabel="Stored points", title="Same current sources, different index representations")
lesson.figure(fig)
plt.show()

## Interpretation and exercise
The private point remains stored but becomes ineligible immediately through current policy. The deleted public source disappears from both indexes after replay. The API also rejects stale source revisions using authoritative source hashes. A Qdrant alias transaction changes routing atomically inside Qdrant; it does not atomically update policy, model servers, caches, or application configuration.

**Exercise:** pause the old consumer after cutover, append a deletion, and attempt rollback. What readiness gate must fail? Design a checkpoint that includes source-log position, representation signature, chunking signature, and policy epoch. Explain why rolling back an application must never roll back a permission revocation.

This lab drains and recreates the service at each switch, producing a bounded maintenance window. A zero-downtime design needs immutable per-request index/model references and coordinated routing generations. Retain the old index only for a declared rollback window, and account for dual-write, cache, disk, and index-build costs. [Qdrant collection aliases](https://qdrant.tech/documentation/concepts/collections/#collection-aliases).

In [ ]:
manifest = lesson.finish(
    {
        "interruption_recovered": interrupted,
        "stale_source_write_rejected": stale_rejected,
        "both_manifests_reconcile": reconciled,
        "cutover_revision_current": new_keys == {"manual@2"},
        "rollback_revision_current": rollback_keys == {"manual@3"},
        "rollback_alias_correct": alias_target == old.name,
        "new_chunking_differs": point_counts[1] > point_counts[0],
    },
    observations={
        "backfill_watermark": watermark,
        "final_journal_head": head,
        "cutover": "drained service replacement plus Qdrant alias transaction",
        "policy": "current authoritative store retained across both versions",
        "single_writer": True,
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.